In [18]:
# 1. YOUR AUTHENTICATED TOKEN
NGROK_TOKEN = "Paste Ngrok AuthToken"

# 2. INSTALL EVERYTHING (Using Google Chrome directly to bypass Snaps)
import os, time
print("📦 Installing Desktop and Chrome... please wait.")
!killall -9 x11vnc fluxbox xvfb firefox google-chrome ngrok tint2 > /dev/null 2>&1
!rm -rf noVNC

# Install core desktop
!apt-get update -qq
!apt-get install -y x11vnc fluxbox xvfb tint2 dbus-x11 -qq

# Download and install Google Chrome directly
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -qq

!pip install pyngrok pyvirtualdisplay --quiet
!git clone https://github.com/novnc/noVNC.git --quiet

# 3. START VIRTUAL DISPLAY
from pyvirtualdisplay import Display
from pyngrok import ngrok, conf
display = Display(visible=0, size=(1024, 768))
display.start()
os.environ["DISPLAY"] = ":0"
os.environ["DBUS_SESSION_BUS_ADDRESS"] = "autolaunch:"

# 4. LAUNCH DESKTOP SERVICES
print("🖥️ Starting Desktop...")
os.system("dbus-launch fluxbox &")
os.system("tint2 &")
os.system("x11vnc -display :0 -nopw -listen localhost -xkb -forever &")

# 5. START WEB BRIDGE
os.system("nohup ./noVNC/utils/novnc_proxy --vnc localhost:5900 --listen localhost:6080 > /dev/null 2>&1 &")

# 6. LAUNCH CHROME (With strict container-bypass flags)
print("🌐 Launching Chrome...")
chrome_flags = (
    "--no-sandbox "                  # Crucial for Colab/Docker
    "--disable-dev-shm-usage "       # Prevents crashes from limited shared memory
    "--disable-gpu "                 # Disables hardware acceleration
    "--disable-software-rasterizer " # Stops the renderer from hanging
    "--disable-setuid-sandbox "      # Disables secondary sandbox
    "--user-data-dir=/tmp/chrome-profile"
)
os.system(f"google-chrome {chrome_flags} https://www.google.com &")

# 7. START NGROK
print("🔗 Connecting Tunnel...")
conf.get_default().auth_token = NGROK_TOKEN
try:
    ngrok.kill()
    time.sleep(2)
    public_url = ngrok.connect(6080, bind_tls=True)

    print("\n" + "🚀" * 15)
    print(f"URL: {public_url.public_url}/vnc.html")
    print("🚀" * 15 + "\n")

    # KEEP ALIVE
    while True:
        time.sleep(60)

except Exception as e:
    print(f"❌ Error: {e}")

📦 Installing Desktop and Chrome... please wait.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
🖥️ Starting Desktop...
🌐 Launching Chrome...
🔗 Connecting Tunnel...

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
URL: https://ce82-34-86-234-157.ngrok-free.app/vnc.html
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀



KeyboardInterrupt: 